# LingkodAI golden check on Colab

Runs the full seven-stage pipeline on the three golden conversations (one per
language, `CONV-MT-001`) and compares every text field against DS6, the
evaluated Scenario 2 run. This is the same `scripts/golden_check.py` that runs
on JOJIE; see the README's Testing section for how to read the report.

**Before you start**

- Runtime: GPU. **L4 preferred** (native bf16, like the original RAG Colab
  run). A T4 is untested but should fit: it has more memory than the 11 GB
  cards the pipeline runs on at JOJIE. The pipeline runs `staged`, so one model family is on
  the GPU at a time.
- A Colab Secret named `HF_TOKEN` (key icon in the left sidebar) with notebook
  access on. It must be a fine-grained read-only token from your own Hugging
  Face account, as a `lingkodai` org member, that can read the three ASR model
  repos and the `lingkodai/lingkodai-eval-assets` dataset repo.
- Roughly 45 GB of model downloads plus the eval assets. Expect the first run
  to spend a good while downloading.

The TTS reference clips are Philippine Languages Database speaker audio under
a CC-BY-NC, research-only licence, and the synthesized voice is a clone of
that speaker. Do not share the downloaded clips or the output WAVs publicly.

Run the cells in order.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU (L4 preferred)"
print(torch.cuda.get_device_name(0), "| arch list:", torch.cuda.get_arch_list())

## 2. Get the code and install

Uses Colab's preinstalled torch and torchaudio (built for this runtime's CUDA)
and installs the rest from `pyproject.toml`. `qwen-tts` pins
`transformers==4.57.3` and `accelerate==1.12.0`; pip may downgrade Colab's
copies to match, which is expected. `pip check` may also list conflicts
between unrelated packages Colab preinstalls; only a conflict involving
`transformers`, `accelerate`, `qwen-tts` or `torch` matters here. Every later
step runs as a separate `python` process, so no runtime restart is needed. If
Colab offers "Restart session" anyway, either choice is fine: every later cell
moves back into the repo folder first, so just carry on with step 3.

In [ ]:
BRANCH = "main"
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg sox > /dev/null
import os
!test -d /content/LingkodAI || git clone -q --branch {BRANCH} https://github.com/isabelbeab/LingkodAI.git /content/LingkodAI
assert os.path.isdir("/content/LingkodAI/scripts"), "git clone failed; see the output above"
%cd /content/LingkodAI
!git log --oneline -1
!pip install -q -e ".[gpu,eval]"
assert _exit_code == 0, "pip install failed; see the output above"
!pip check
!python -c "import torch, torchaudio, transformers, accelerate; print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| transformers', transformers.__version__, '| accelerate', accelerate.__version__)"

## 3. Token and environment

Reads `HF_TOKEN` from Colab Secrets into this kernel's environment, which the
scripts below inherit. The token is never printed or written to disk.

Also mounts Google Drive now, for step 7. Mounting asks for permission in a
popup; doing it here means the long run below never waits on a popup that
nobody is watching.

In [ ]:
%cd /content/LingkodAI
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
assert os.environ["HF_TOKEN"], "The HF_TOKEN secret is empty"
print("HF_TOKEN set")

from google.colab import drive
drive.mount("/content/drive")

## 4. Patch the ASR tokenizers

The fine-tuned Whisper checkpoints do not load as published under
transformers 4.57.3 (`CHANGELOG.md`, 2026-09-22). This builds verified,
patched local copies (weights symlinked, not duplicated) and points
`src/asr.py` at them. Any verification mismatch raises.

In [ ]:
%cd /content/LingkodAI
!python scripts/patch_asr_tokenizers.py --out outputs/asr_patched
assert _exit_code == 0, "patch_asr_tokenizers.py failed; see the output above"
for lang in ("ceb", "fil", "eng"):
    path = f"/content/LingkodAI/outputs/asr_patched/{lang}"
    assert os.path.isdir(path), f"missing {path}: the patch step failed"
    os.environ[f"ASR_{lang.upper()}"] = path
    print(f"ASR_{lang.upper()}={path}")

## 5. Fetch the private eval assets

Downloads the TTS reference clips, DS6 and the 24 golden clips, and checks
every file against the repo's `MANIFEST.sha256`. Stops here if anything is
missing or changed.

In [ ]:
%cd /content/LingkodAI
ASSETS = "/content/LingkodAI/outputs/eval_assets"
!python scripts/fetch_eval_assets.py --out {ASSETS}
assert _exit_code == 0, "fetch_eval_assets.py failed; see the output above"
os.environ["TTS_REF_DIR"] = f"{ASSETS}/tts_ref"
print("TTS_REF_DIR =", os.environ["TTS_REF_DIR"])

## 6. Run the golden check

Three conversations, eight turns each, every model loaded once per
conversation. This is the long step. The log is also written to
`outputs/golden_check.log`.

In [ ]:
%cd /content/LingkodAI
ASSETS = "/content/LingkodAI/outputs/eval_assets"
DS6_AUDIO_ROOT = "/home2/msds2026/ibucayan/Capstone/PRC_synthetic_qa"
cmd = (
    f"python scripts/golden_check.py --ds6 {ASSETS}/golden/DS6_augmented.xlsx"
    f" --conversation-id CONV-MT-001 --audio-root-map {DS6_AUDIO_ROOT}={ASSETS}/golden/audio"
    " --out outputs/golden_check"
)
print(cmd)
!{cmd} 2>&1 | tee outputs/golden_check.log

## 7. Save the results to Google Drive

Colab's disk is wiped when the runtime ends, so run this as soon as step 6
finishes. This copies the report, per-turn JSON, WAVs and log (not the models or eval assets) to
`MyDrive/lingkodai_golden_check/<timestamp>/`. Keep that folder private.

In [ ]:
%cd /content/LingkodAI
import shutil, time

assert os.path.isdir("/content/drive/MyDrive"), "Drive is not mounted: rerun step 3's cell"
dest = f"/content/drive/MyDrive/lingkodai_golden_check/{time.strftime('%Y%m%d-%H%M%S')}"
shutil.copytree("outputs/golden_check", dest)
shutil.copy("outputs/golden_check.log", dest)
print("saved to", dest)

## 8. Listen

TTS is judged by listening, not by the script. One player per turn that
synthesized; a turn with no WAV fell back to text only, and its reason is in
that turn's JSON.

In [ ]:
%cd /content/LingkodAI
from pathlib import Path
from IPython.display import Audio, Markdown, display

for wav in sorted(Path("outputs/golden_check").glob("*.wav")):
    display(Markdown(f"**{wav.stem}**"))
    display(Audio(str(wav)))